In [ ]:
pip install SpeechRecognition pyaudio requests beautifulsoup4


In [ ]:
import speech_recognition as sr
import requests
from bs4 import BeautifulSoup

# PUT YOUR TOKEN HERE
GENIUS_API_KEY = "laUh5csjovXsHIQaTL-x7w9TLr72t_ytaNj2XYtWTpmFMBWNgIeNAlvhVOU6LfDX"

# Step 1: Speech Recognition
def recognize_speech():
    recognizer = sr.Recognizer()

    with sr.Microphone() as source:
        print("Say a song name...")
        recognizer.adjust_for_ambient_noise(source)
        audio = recognizer.listen(source)

    try:
        text = recognizer.recognize_google(audio)
        print("You said:", text)
        return text
    except sr.UnknownValueError:
        print("Could not understand audio")
        return None
    except sr.RequestError:
        print("Speech service unavailable")
        return None


# Step 2: Search Genius
def search_song(song_name):
    url = "https://api.genius.com/search"
    headers = {
        "Authorization": f"Bearer {GENIUS_API_KEY}"
    }
    params = {"q": song_name}

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    hits = data["response"]["hits"]

    if hits:
        song = hits[0]["result"]
        return {
            "title": song["title"],
            "artist": song["primary_artist"]["name"],
            "url": song["url"]
        }
    else:
        return None


# Step 3: Scrape Lyrics
def get_lyrics(song_url):
    page = requests.get(song_url)
    html = BeautifulSoup(page.text, "html.parser")

    # Genius uses special divs for lyrics
    lyrics_divs = html.find_all("div", {"data-lyrics-container": "true"})

    lyrics = ""
    for div in lyrics_divs:
        lyrics += div.get_text(separator="\n")

    return lyrics.strip()


# MAIN PROGRAM
def main():
    spoken_text = recognize_speech()

    if not spoken_text:
        return

    song = search_song(spoken_text)

    if not song:
        print("Song not found")
        return

    print(f"\n🎵 Found: {song['title']} by {song['artist']}")
    print("🔗", song["url"])

    print("\n📄 Fetching lyrics...\n")

    lyrics = get_lyrics(song["url"])

    # Print only first part to avoid huge output
    print(lyrics[:1500])


if __name__ == "__main__":
    main()